# 🚀 TinyTransformer-238M: Ultra High-Speed T4 GPU Pretraining on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/logicstillloading/TinyTransformer/blob/main/TinyTransformer_Colab_T4.ipynb)

Pretraining the **~237.8M Parameter TinyTransformer** architecture featuring:
- **Reasoning-After-Retrieval Dynamic**: 80M global Engram table (post-Layer 16) + two 10M local Engram tables (Layers 5 & 11) followed by a 7-layer post-retrieval reasoning block.
- **Keller Jordan's Muon Optimizer**: 5th-order Newton-Schulz orthogonalization for 2D weight matrices + AdamW for embeddings, RMSNorms, and sparse Engram tables.
- **Adversarial Critic RL Co-Training**: Auxiliary ~9.2M parameter discriminator detecting hallucinations and rewarding crisp, grounded generations.

---

### ⚡ Maximum Throughput (Tokens/Sec Maxing) on Tesla T4:
- **Turing FP16 Tensor Cores (130 TFLOPs)**: Tesla T4 lacks native hardware BF16. This script automatically detects T4 and enables FP16 mixed precision with dynamic `GradScaler`, yielding **4x-5x faster throughput** than BF16 CPU emulation.
- **16 GB VRAM Parallel Saturation**: Scaled from laptop batch size 1 to **Micro-batch 4 (or 8)** with sequence length 1024, maximizing GPU core occupancy.
- **Google Drive Checkpoint Sync**: Atomic checkpointing rotates directly into Google Drive (`/content/drive/MyDrive/TinyTransformer/checkpoints`), so **zero progress is lost** even if Colab disconnects!


## 1. Verify NVIDIA GPU Hardware & Memory

In [ ]:
# Check GPU device name and allocated VRAM
!nvidia-smi

import torch
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:        {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM:       {vram_gb:.2f} GB")
    print(f"BF16 Supported:   {torch.cuda.is_bf16_supported()}")
    if not torch.cuda.is_bf16_supported():
        print("[*] Tesla T4 detected: Hardware FP16 Tensor Cores will be utilized for maximum tokens/sec throughput!")


## 2. Mount Google Drive for Continuous Atomic Checkpointing
Mounting Google Drive guarantees that checkpoints (`checkpoint_latest.pt`, `checkpoint_prev.pt`, and milestone archives) and session logs are saved directly to your Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/TinyTransformer'
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
LOG_DIR = os.path.join(DRIVE_DIR, 'logs')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f'[+] Google Drive mounted. Checkpoints will be safely preserved at: {CHECKPOINT_DIR}')


### 2b. (Optional) Resume from Existing Checkpoint (~256M tokens / Step 7,811)
If you want to resume your previous training run rather than starting from Step 0, you can either:
1. Drag & drop `checkpoint_latest.pt` directly into your Google Drive folder: `MyDrive/TinyTransformer/checkpoints/checkpoint_latest.pt`, OR
2. Uncomment and run the code below to upload it directly from your computer into Drive.

In [ ]:
# from google.colab import files
# import shutil
# print('Select checkpoint_latest.pt from your PC:')
# uploaded = files.upload()
# for fn in uploaded.keys():
#     if fn.endswith('.pt'):
#         dest = os.path.join(CHECKPOINT_DIR, 'checkpoint_latest.pt')
#         shutil.move(fn, dest)
#         print(f'[+] Checkpoint placed at: {dest}')


## 3. Clone TinyTransformer Repository & Install Dependencies

In [ ]:
# Clone latest repository
!rm -rf TinyTransformer
!git clone https://github.com/logicstillloading/TinyTransformer.git
%cd TinyTransformer

# Install required lightweight dependencies
!pip install -r requirements.txt -q
print('[+] Dependencies installed successfully!')


## 4. Architectural Verification & Parameter Breakdown (~238M)

In [ ]:
from model.transformer import TinyTransformer, TransformerConfig
from data.tokenizer import get_or_create_tokenizer

tok = get_or_create_tokenizer('tokenizer.json')
cfg = TransformerConfig(vocab_size=tok.vocab_size)
model = TinyTransformer(cfg)
bd = model.get_parameter_breakdown()

print('=' * 60)
print(f"  Total Parameters:    {bd['total_params']/1e6:.2f}M ({bd['unique_params']:,} unique)")
print(f"  Backbone Parameters: {bd['backbone_params']/1e6:.2f}M")
print(f"  Engram Memory:       {bd['engram_params']/1e6:.2f}M (80M post-L16 + two 10M at L5/L11)")
print(f"  Model Footprint:     {bd['bf16_weight_size_mb']:.1f} MB")
print('=' * 60)
del model


## 5. Launch High-Throughput T4 GPU Pretraining 🚀

This launch command maximizes GPU throughput by:
- Using `--batch-size 4 --grad-accum 8` (effective 32,768 tokens/step) for high parallel tensor utilization in 16GB VRAM.
- Auto-detecting `--dtype auto` to run hardware-accelerated FP16 with `GradScaler` on T4.
- Pointing `--checkpoint-dir` and `--log-dir` directly to Google Drive so training resumes seamlessly across sessions.

In [ ]:
# Start or resume pretraining (Auto-profiles for T4, TPU v5e, or RTX 3050 Ti!)
!python train.py \
    --checkpoint-dir /content/drive/MyDrive/TinyTransformer/checkpoints \
    --log-dir /content/drive/MyDrive/TinyTransformer/logs \
    --lr-muon 0.006 \
    --lr-adam 0.0003 \
    --lr-warmup-steps 50 \
    --lr-decay-steps 79000 \
    --save-interval 250 \
    --infer-interval 250 \
    --archive-interval 2500 \
    --resume


## 6. Run GPU Factual Probes and Generation Benchmarks
Evaluate the trained checkpoint on GPU to check vocabulary ranking on factual probes (Einstein, Washington, Paris, H2O) and generate sample completions.

In [ ]:
!python test.py \
    --checkpoint /content/drive/MyDrive/TinyTransformer/checkpoints/checkpoint_latest.pt \
    --device cuda \
    --max-new-tokens 60 \
    --temperature 0.70


## 7. Interactive Prompt Playground
Test custom queries interactively inside the notebook.

In [ ]:
import os
import torch
from data.tokenizer import get_or_create_tokenizer
from model.transformer import TinyTransformer, TransformerConfig

tok = get_or_create_tokenizer('tokenizer.json')
cfg = TransformerConfig(vocab_size=tok.vocab_size)
model = TinyTransformer(cfg).cuda().eval()

ckpt_path = '/content/drive/MyDrive/TinyTransformer/checkpoints/checkpoint_latest.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cuda', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    print(f"[+] Loaded checkpoint at global step {ckpt.get('step', 0):,} ({ckpt.get('tokens_processed', 0)/1e6:.1f}M tokens)")
else:
    print('[!] No checkpoint found in Drive yet. Initialized with fresh weights.')

def generate(prompt: str, max_tokens: int = 80, temp: float = 0.75):
    input_ids = torch.tensor([tok.encode(prompt, add_bos=True)], device='cuda')
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=max_tokens, temperature=temp, top_k=40, top_p=0.90)
    return tok.decode(out[0].tolist(), skip_special_tokens=True)

# Test prompts
print(generate('User: Explain how gravity works.\nAssistant:'))
print('-' * 50)
print(generate('def quicksort(arr):\n    """Sort list recursively."""\n'))


## 8. Visualize Loss and Perplexity Curves
Plot training loss, EMA loss, and step perplexity directly inline.

In [ ]:
%matplotlib inline
!python plot_metrics.py --log-dir /content/drive/MyDrive/TinyTransformer/logs


## 9. Safe Remote Control Flags (Non-Destructive Pause & Snapshot)
Run any of the commands below to interactively signal the training process without stopping abruptly:

In [ ]:
# Graceful stop: completes current step, saves latest checkpoint, and exits cleanly
# !touch stop.flag

# Immediate save: writes an atomic snapshot without stopping
# !touch save.flag

# Cancel flags
# !rm -f stop.flag save.flag critic.flag


## 10. Sync Checkpoint to Hugging Face Hub (Cloud-to-Local Bridge)
Publish your latest checkpoint to Hugging Face Hub so your local PC (`train.py --resume --cloud-sync`) can automatically pull it and continue training seamlessly:

In [ ]:
# Push latest checkpoint to Hugging Face:
# !python data/cloud_sync.py --push --hf-repo your-username/TinyTransformer-checkpoints \
#     --checkpoint-dir /content/drive/MyDrive/TinyTransformer/checkpoints
